# B03 — Type 1 Parser Eval (via `/parser` API)

Sends each inference instance's `premises-NL` to the EXACT API `/parser` endpoint
and inspects the returned FOL translations.

- **Input:** `Logic_Based_Educational_Queries_inference.json` — only the `premises-NL` field is used.
- **Output:** FOL string + AST per premise, returned by the parser vLLM.
- **Scoring:** parse success rate + latency (no answer label comparison here).


In [1]:
import json, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

# --- endpoint -------------------------------------------------------------
API_BASE   = "https://api.iamphuckhang.dev"   # VM via cloudflared tunnel
# API_BASE = "http://127.0.0.1:8080"           # only if kernel runs ON the VM
PARSER_URL = f"{API_BASE}/parser"

# --- run size -------------------------------------------------------------
N_SAMPLES   = 20        # how many instances to eval (None = all 808)
CONCURRENCY = 8         # parallel in-flight requests
TIMEOUT     = 120.0     # per-request seconds

# --- locate dataset dir (walk up to project root) -------------------------
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", PARSER_URL)

dataset : /home/phuckhang/MyWorkspace/Exact2026/src/exact/datasets/exact
endpoint: https://api.iamphuckhang.dev/parser


In [2]:
inf = json.load(open(DATA / "Logic_Based_Educational_Queries_inference.json"))["instances"]
print(f"{len(inf)} inference instances")


def build_request(inst: dict) -> dict:
    # /parser only needs the premises; field name matches the AliasChoices on the server.
    return {"premises": inst["premises-NL"]}


# sanity-check on first instance
ex = inf[0]
req = build_request(ex)
print(f"\nInstance : {ex['id']}")
print(f"Premises : {len(req['premises'])}")
for i, p in enumerate(req["premises"], 1):
    print(f"  [{i}] {p}")

808 inference instances

Instance : logic_0000_00
Premises : 14
  [1] If a Python code is well-tested, then the project is optimized.
  [2] If a Python code does not follow PEP 8 standards, then it is not well-tested.
  [3] All Python projects are easy to maintain.
  [4] All Python code is well-tested.
  [5] If a Python code follows PEP 8 standards, then it is easy to maintain.
  [6] If a Python code is well-tested, then it follows PEP 8 standards.
  [7] If a Python project is well-structured, then it is optimized.
  [8] If a Python project is easy to maintain, then it is well-tested.
  [9] If a Python project is optimized, then it has clean and readable code.
  [10] All Python projects are well-structured.
  [11] All Python projects have clean and readable code.
  [12] There exists at least one Python project that follows best practices.
  [13] There exists at least one Python project that is optimized.
  [14] If a Python project is not well-structured, then it does not follow PEP 8 s

In [3]:
async def call(client, sem, inst):
    payload = build_request(inst)
    async with sem:
        t0 = time.perf_counter()
        err, parsed = None, []
        try:
            r = await client.post(PARSER_URL, json=payload, timeout=TIMEOUT)
            r.raise_for_status()
            parsed = r.json().get("premises", [])
        except Exception as e:
            err = repr(e)
        dt = time.perf_counter() - t0
    return {
        "id": inst["id"],
        "parsed": parsed,
        "n_premises": len(inst["premises-NL"]),
        "n_parsed": len(parsed),
        "latency": dt,
        "error": err,
    }


async def run_eval(instances):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(inst):
            nonlocal done
            res = await call(client, sem, inst)
            done += 1
            if done % 10 == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(i) for i in instances))

In [12]:
subset = inf[:N_SAMPLES] if N_SAMPLES else inf
print(f"Parsing premises for {len(subset)} instances at concurrency {CONCURRENCY}...")
t0 = time.perf_counter()
results = await run_eval(subset)    # Jupyter supports top-level await
wall = time.perf_counter() - t0

errors  = [r for r in results if r["error"]]
success = [r for r in results if not r["error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")

Parsing premises for 20 instances at concurrency 8...
  20/20
Success : 20/20
Errors  : 0
Wall    : 31.0s


In [14]:
# --- Errors ---
if errors:
    print("Errors:")
    for r in errors:
        print(f"  [{r['id']}] {r['error']}")
    print()

# --- Sample FOL output from the first successful result ---
for r in success:
    print(f"--- {r['id']} ({r['n_premises']} premises) ---")
    for p in r["parsed"]:
        print(f"  {p['id']}: {p['fol']}")
    print()

--- logic_0000_00 (14 premises) ---
  premise-1: ∀x.(WellTested(x) IMPLIES Optimized(Project))
  premise-2: ∀x.(NOT(Follows(x, PEP8Standards)) IMPLIES NOT(WellTested(x)))
  premise-3: ∀x.(EasyToMaintain(x))
  premise-4: ∀x.(WellTested(x))
  premise-5: ∀x.(Follows(x, PEP8Standards) IMPLIES EasyToMaintain(x))
  premise-6: ∀x.(WellTested(x) IMPLIES Follows(x, PEP8Standards))
  premise-7: ∀x.(WellStructured(x) IMPLIES Optimized(x))
  premise-8: ∀x.(EasyToMaintain(x) IMPLIES WellTested(x))
  premise-9: ∀x.(Optimized(x) IMPLIES Has(x, CleanAndReadableCode))
  premise-10: ∀x.(WellStructured(x))
  premise-11: Have(AllPythonProjects, CleanAndReadableCode)
  premise-12: ∃x.(Follows(x, BestPractices))
  premise-13: ∃x.(Optimized(x))
  premise-14: ∀x.(NOT(WellStructured(x)) IMPLIES NOT(WellStructured(x)))

--- logic_0000_01 (14 premises) ---
  premise-1: ∀x.(WellTested(x) IMPLIES Optimized(Project))
  premise-2: ∀x.(NOT(Follows(x, PEP8Standards)) IMPLIES NOT(WellTested(x)))
  premise-3: ∀x.(EasyTo

In [ ]:
# --- Latency stats ---
lat = [r["latency"] for r in success]
if lat:
    print(f"latency  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")

# --- Premise coverage ---
n_sent   = sum(r["n_premises"] for r in results)
n_parsed = sum(r["n_parsed"]   for r in results)
print(f"premises {n_parsed}/{n_sent} returned")

# --- Premise count distribution ---
print("premise count dist:", Counter(r["n_premises"] for r in results))

## Notes
- Set `N_SAMPLES = None` to run all 808 instances.
- Each FOL result has `id`, `original_text`, `fol` (string repr), and `ast` (dict tree).
- To inspect a single result: `next(r for r in results if not r['error'])['parsed'][0]`.
- To see the raw AST: `results[0]['parsed'][0]['ast']`.
